# Part 1.2 — Epsilon-Ball Graph on House Nodes

Laplacian Eigenmaps require an **adjacency matrix**. This dataset has no native
graph, so we synthesise one over the house nodes by linking houses that are close
together.

We use an **epsilon-ball** rather than K-NN: K-NN forces every house to have exactly
K neighbours regardless of local density, which is a poor fit here (a house may
genuinely have many nearby neighbours or very few). An epsilon-ball links any two
houses within a fixed **walking-distance** cutoff, so degree reflects real local
density.

Distance follows the project default: **Haversine, in km**. The cutoff is specified
as a *walking* distance and converted to Haversine via the detour factor (see
PLAN.md), since straight-line distance underestimates true walking distance.

This notebook:
1. Loads the training house dataset and the MRT dataset.
2. Builds an epsilon-ball graph over the houses (default 0.5 km walking distance).
3. Plots the house graph (blue) together with the MRT graph (red) to help tune the
   epsilon cutoff visually.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy.sparse import triu
from sklearn.neighbors import radius_neighbors_graph

# Haversine distances come back in radians on the unit sphere; scale by Earth radius for km.
EARTH_RADIUS_KM = 6371.0088

# Walking paths detour around roads/blocks, so walking distance exceeds straight-line
# (Haversine) distance. Divide the walking cutoff by this factor to get the Haversine radius.
DETOUR_FACTOR = 1.35

# Approximate lat/long bounding box of mainland Singapore
SG_LAT_RANGE = (1.20, 1.48)
SG_LON_RANGE = (103.60, 104.10)

## 1. Load the house and MRT datasets

In [2]:
TRAIN_HOUSE_PATH = "graph_feature_store/train-house.csv"
STATION_KEYS_PATH = "graph_feature_store/station-keys.csv"
MRT_EDGES_PATH = "raw_graph_data/sg-mrt-edges-weighted.csv"

train_house = pd.read_csv(TRAIN_HOUSE_PATH)
stations = pd.read_csv(STATION_KEYS_PATH)
mrt_edges = pd.read_csv(MRT_EDGES_PATH)

print(f"Houses   : {len(train_house):,}")
print(f"Stations : {len(stations)}")
print(f"MRT edges: {len(mrt_edges)}")

Houses   : 150,000
Stations : 192
MRT edges: 224


## 2. Build the epsilon-ball graph over houses

Two houses are linked if within **`EPS_WALK_KM`** of each other (walking distance).
The walking cutoff is converted to a Haversine radius using `DETOUR_FACTOR`, then to
radians for the `haversine` metric.

`radius_neighbors_graph` returns a sparse symmetric adjacency matrix `A` — exactly
the input Laplacian Eigenmaps needs. We report edge count, mean degree, and isolated
houses, which are the quantities to watch while tuning the cutoff.

In [6]:
# --- Tunable parameter --------------------------------------------------
EPS_WALK_KM = 0.35          # epsilon-ball cutoff as a WALKING distance (km)
# ------------------------------------------------------------------------

eps_haversine_km = EPS_WALK_KM / DETOUR_FACTOR   # straight-line equivalent
eps_radians = eps_haversine_km / EARTH_RADIUS_KM

coords_rad = np.radians(train_house[["LATITUDE", "LONGITUDE"]].to_numpy())

# Sparse, symmetric connectivity matrix (self-loops excluded)
A = radius_neighbors_graph(
    coords_rad,
    radius=eps_radians,
    mode="connectivity",
    metric="haversine",
    include_self=False,
)

degrees = np.asarray(A.sum(axis=1)).ravel()
n_edges = int(A.nnz // 2)  # symmetric matrix double-counts each undirected edge
n_isolated = int((degrees == 0).sum())

print(f"EPS_WALK_KM = {EPS_WALK_KM} km  ->  Haversine radius = {eps_haversine_km:.3f} km")
print(f"Edges          : {n_edges:,}")
print(f"Mean degree    : {degrees.mean():.1f}")
print(f"Median degree  : {np.median(degrees):.0f}")
print(f"Min non-zero degree     : {int(degrees[np.where(degrees != 0)].min())}")
print(f"Max degree     : {int(degrees.max())}")
print(f"Isolated houses: {n_isolated:,} ({n_isolated / len(train_house):.1%})")

EPS_WALK_KM = 0.35 km  ->  Haversine radius = 0.259 km
Edges          : 26,769,851
Mean degree    : 356.9
Median degree  : 334
Min non-zero degree     : 3
Max degree     : 1200
Isolated houses: 0 (0.0%)


## 3. Plot the house graph (blue) with the MRT graph (red)

House nodes and their epsilon-ball edges are drawn in **blue**; MRT stations and
their track edges in **red**, matching the DBSCAN notebook's overlay.

A dense epsilon-ball over ~15k urban houses can produce a very large number of
edges — far more than a browser can render smoothly. The edge *count* printed above
is the real signal for tuning; for the plot we cap the number of house edges drawn
(`max_display_edges`) and randomly sample if the graph exceeds it, printing a note
when that happens.

In [ ]:
def _edge_segments(lon, lat, ij_pairs):
    """Flatten (i, j) index pairs into None-separated x/y arrays for a single line trace."""
    xs, ys = [], []
    for i, j in ij_pairs:
        xs += [lon[i], lon[j], None]
        ys += [lat[i], lat[j], None]
    return xs, ys


def plot_epsilon_graph(
    house_df,
    house_adj,
    stations,
    mrt_edges,
    title="Epsilon-ball house graph with MRT network",
    lat_range=SG_LAT_RANGE,
    lon_range=SG_LON_RANGE,
    max_display_edges=50_000,
    seed=0,
):
    """Plot houses + epsilon-ball edges (blue) with MRT stations + edges (red).

    house_adj : sparse symmetric adjacency over the rows of house_df.
    max_display_edges : cap on house edges drawn; sampled at random if exceeded.
    """
    h_lon = house_df["LONGITUDE"].to_numpy()
    h_lat = house_df["LATITUDE"].to_numpy()

    # Unique undirected house edges from the upper triangle
    upper = triu(house_adj, k=1).tocoo()
    rows, cols = upper.row, upper.col
    n_total = len(rows)
    if n_total > max_display_edges:
        rng = np.random.default_rng(seed)
        pick = rng.choice(n_total, size=max_display_edges, replace=False)
        rows, cols = rows[pick], cols[pick]
        print(
            f"Drawing {max_display_edges:,} of {n_total:,} house edges (random sample) "
            f"to keep the plot responsive."
        )
    hx, hy = _edge_segments(h_lon, h_lat, zip(rows, cols))

    fig = px.scatter(
        title=title,
        labels={"x": "Longitude", "y": "Latitude"},
    )

    # House edges (blue), then house nodes (blue) on top
    fig.add_scatter(x=hx, y=hy, mode="lines", line=dict(color="royalblue", width=0.5),
                    opacity=0.4, name="house edges")
    fig.add_scatter(x=h_lon, y=h_lat, mode="markers",
                    marker=dict(color="royalblue", size=4), opacity=0.6, name="houses")

    # MRT edges (red), then stations (red) on top
    coord = stations.set_index("NAME")[["latitude", "longitude"]]
    m_lon = stations["longitude"].to_numpy()
    m_lat = stations["latitude"].to_numpy()
    mrt_pairs, missing = [], 0
    name_to_pos = {name: k for k, name in enumerate(stations["NAME"])}
    for src, dst in zip(mrt_edges["src_station"], mrt_edges["dst_station"]):
        if src in name_to_pos and dst in name_to_pos:
            mrt_pairs.append((name_to_pos[src], name_to_pos[dst]))
        else:
            missing += 1
    if missing:
        print(f"{missing} MRT edge(s) skipped — station name not found in station-keys")
    mx, my = _edge_segments(m_lon, m_lat, mrt_pairs)

    fig.add_scatter(x=mx, y=my, mode="lines", line=dict(color="red", width=1),
                    opacity=0.8, name="MRT lines")
    fig.add_scatter(x=m_lon, y=m_lat, mode="markers", marker=dict(color="red", size=6),
                    name="MRT stations", text=stations["NAME"], hoverinfo="text")

    fig.update_xaxes(range=list(lon_range))
    fig.update_yaxes(range=list(lat_range))
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.update_layout(width=950, height=650, legend_title_text="")
    return fig


fig = plot_epsilon_graph(
    train_house,
    A,
    stations,
    mrt_edges,
    title=f"Epsilon-ball house graph (EPS_WALK_KM={EPS_WALK_KM}) with MRT network",
)
fig.show()

## 4. Laplacian Eigenmaps — spectral decomposition

Steps:
1. **Adjacency** — reuse the sparse `A` from Section 2 (already a scipy sparse
   matrix built from the epsilon graph); symmetrised and cast to float.
2. **Normalised Laplacian** — `L = I - D^{-1/2} A D^{-1/2}` (symmetric normalisation).
3. **Eigen-decomposition** — the `k` smallest non-trivial eigenvectors of `L`,
   with `k` tunable.

**Implementation note — why we decompose `S = D^{-1/2} A D^{-1/2}` instead of `L`.**
The smallest eigenvalues of `L` are the *largest* eigenvalues of `S` (since
`L = I - S`), and `eigsh(..., which='LA')` (largest algebraic) is far more numerically
stable and faster than `which='SM'` on `L`. The eigenvectors are shared, and
`λ = 1 − μ`. As a bonus, this yields the normalised-affinity eigenvalues `μ_k` and
degrees that the **Nyström** out-of-sample extension needs, so nothing is recomputed.

**Trivial mode.** The constant eigenvector (`λ = 0`, `μ = 1`) carries no information
and is dropped. If the graph has more than one connected component there are that
many `λ = 0` modes (component indicators), so we report the component count.

**Cost caveat.** With 150k nodes and a very dense graph (mean degree ~600 at
`EPS_WALK_KM = 0.5`), each Lanczos matvec touches tens of millions of non-zeros —
`eigsh` will be memory- and time-heavy. Reducing `EPS_WALK_KM` (Section 2) shrinks
the graph and speeds this up substantially; tune epsilon down before scaling `k`.

In [ ]:
from scipy.sparse import diags, eye
from scipy.sparse.linalg import eigsh
from scipy.sparse.csgraph import connected_components


def normalized_laplacian(adj):
    """Symmetric normalised Laplacian  L = I - D^{-1/2} A D^{-1/2}.

    Returns (L, S, deg) where S = D^{-1/2} A D^{-1/2} is the normalised affinity
    and deg is the vector of node degrees. Isolated nodes (degree 0) get a 0 row
    in S (no division by zero).
    """
    adj = adj.maximum(adj.T).astype(float)          # enforce exact symmetry
    deg = np.asarray(adj.sum(axis=1)).ravel()
    d_inv_sqrt = np.zeros_like(deg)
    nz = deg > 0
    d_inv_sqrt[nz] = 1.0 / np.sqrt(deg[nz])
    D_inv_sqrt = diags(d_inv_sqrt)
    S = (D_inv_sqrt @ adj @ D_inv_sqrt).tocsr()
    L = (eye(adj.shape[0], format="csr") - S).tocsr()
    return L, S, deg


def laplacian_eigenmaps(adj, k, drop_trivial=True):
    """Bottom-k non-trivial eigenpairs of the symmetric normalised Laplacian.

    Solved via the LARGEST eigenvalues of S = D^{-1/2} A D^{-1/2} (== smallest of
    L = I - S), which is far more stable/faster with eigsh than which='SM' on L.

    Returns:
      V   : (n, k) eigenvector matrix — each ROW is a node's k-dim embedding
      lam : (k,)  Laplacian eigenvalues, ascending  (lam = 1 - mu)
      mu  : (k,)  normalised-affinity eigenvalues    (reused by Nystrom)
      deg : (n,)  node degrees                        (reused by Nystrom)
    """
    _, S, deg = normalized_laplacian(adj)
    n_req = k + 1 if drop_trivial else k             # +1 to discard the trivial mode
    mu, V = eigsh(S, k=n_req, which="LA")            # largest algebraic eigenvalues of S
    order = np.argsort(mu)[::-1]                      # descending mu  => ascending lambda
    mu, V = mu[order], V[:, order]
    if drop_trivial:
        mu, V = mu[1:], V[:, 1:]                      # drop constant mode (mu ~ 1, lambda ~ 0)
    lam = 1.0 - mu
    return V, lam, mu, deg


def stack_embedding(V, index=None, prefix="lap_emb_"):
    """Stack the k eigenvectors into a per-node k-dimensional embedding table.

    Row i is node i's graph encoding on the k smallest non-trivial eigenvectors.
    """
    cols = [f"{prefix}{i + 1}" for i in range(V.shape[1])]
    return pd.DataFrame(V, index=index, columns=cols)

In [ ]:
# --- Tunable parameter --------------------------------------------------
K_EIG = 3          # number of smallest non-trivial eigenvectors to keep
# ------------------------------------------------------------------------

# Connectivity check: a disconnected graph has one lambda=0 mode per component,
# so the leading modes would encode component membership rather than smooth geometry.
n_components, _ = connected_components(A, directed=False)
if n_components > 1:
    print(f"WARNING: graph has {n_components} connected components — the first "
          f"{n_components} near-zero Laplacian modes are component indicators. "
          f"Consider raising EPS_WALK_KM or dropping more trivial modes.")
else:
    print("Graph is fully connected (1 component).")

# Bottom-k non-trivial eigenpairs (mu and deg are kept for the Nystrom step)
V, lam, mu, deg = laplacian_eigenmaps(A, k=K_EIG)
print(f"\nLaplacian eigenvalues (ascending): {np.round(lam, 5)}")

# Stack into a per-node k-dimensional embedding table
house_embedding = stack_embedding(V, index=train_house.index)
print(f"Embedding shape: {house_embedding.shape}")
house_embedding.head()